In [1]:
import cv2, os, glob, torch
import numpy as np
from torchvision import transforms
from PIL import Image

In [2]:
detect = cv2.CascadeClassifier("haarcascade_frontalface_default.xml")

### Define the PyTorch model architecture

In [3]:
class EmotionCNN(torch.nn.Module):
    def __init__(self, num_classes):
        super(EmotionCNN, self).__init__()
        self.conv_layers = torch.nn.Sequential(
            torch.nn.Conv2d(1, 32, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=2, stride=2),
            
            torch.nn.Conv2d(32, 64, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=2, stride=2),
            
            torch.nn.Conv2d(64, 128, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.fc_layers = torch.nn.Sequential(
            torch.nn.Flatten(),
            torch.nn.Linear(128 * 6 * 6, 256),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.5),
            torch.nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

In [4]:
emotion_labels = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']

# Instantiate the model
num_classes = len(emotion_labels)
emotion_model = EmotionCNN(num_classes=num_classes)

### Load the pre-trained weights

In [5]:
emotion_model.load_state_dict(torch.load("emotion_model.pth", map_location=torch.device("cpu")))
emotion_model.eval()

EmotionCNN(
  (conv_layers): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc_layers): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4608, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=256, out_features=7, bias=True)
  )
)

### Define image transformations

In [6]:
transform = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # Normalize to [-1, 1]
])

### Define the folder to save processed images

In [9]:
import os, shutil

output_folder = "processed_images"

# Check if the folder exists then Remove the folder and its contents
if os.path.exists(output_folder):
    shutil.rmtree(output_folder) 

# Create a new folder
os.makedirs(output_folder)

----------
## Emotion Detection

In [10]:
globimg = glob.glob("jpg images/*.jpg")

for timage in globimg:
    image = cv2.imread(timage)
    
    height, width = image.shape[:2]
    max_width = 500
    if width > max_width:
        scale = max_width / width
        image = cv2.resize(image, (int(width * scale), int(height * scale)))

    grayimg = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = detect.detectMultiScale(grayimg, scaleFactor=1.25, minNeighbors=3, minSize=(30, 30))

    for (x, y, w, h) in faces:
        # Extract the face region
        face_roi = grayimg[y:y+h, x:x+w]
        face_pil = Image.fromarray(face_roi)  # Convert to PIL image for transforms
        face_tensor = transform(face_pil).unsqueeze(0)  # Apply transformations and add batch dimension

        # Predict emotion using the PyTorch model
        with torch.no_grad():
            emotion_prediction = emotion_model(face_tensor)
        emotion_label = emotion_labels[torch.argmax(emotion_prediction).item()]

        # Draw rectangle around the face
        cv2.rectangle(image, (x, y), (x+w, y+h), (0, 255, 0), 2)
        
        # Display emotion label above the face
        cv2.putText(image, emotion_label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    # Save the annotated image to the output folder
    output_path = os.path.join(output_folder, os.path.basename(timage))
    cv2.imwrite(output_path, image)

    cv2.imshow("Emotion Detection", image)
    cv2.waitKey(2000)
    cv2.destroyAllWindows()